# 02 — Ajustes nos dados

**CRISP-DM fase 3 — preparação dos dados** · Retenção de clientes Telecom

---

A [análise exploratória](01-analise-exploratoria.ipynb) descreveu a base e gravou, ao final, 
as colunas selecionadas para a modelagem (`Telcon-Customer-Churn_columns_selected.csv`):
7.043 clientes, 19 colunas, já sem `customerID` e sem `Churn`. A EDA apontou o que ainda
precisa ser corrigido: 11 células vazias em `TotalCharges`, categorias que repetem a
informação de outra coluna (`No internet service`, `No phone service`) e uma variável
binária (`SeniorCitizen`) codificada como número. Este notebook é o passo seguinte:
**aplicar as correções** e gravar a base curada que as fases posteriores consomem.

A regra deste notebook é uma só:

> **Nenhum dado é alterado em silêncio.** Cada ajuste declara o *motivo*, mostra a
> *evidência numérica* que o justifica e registra *quantos registros* foram tocados. 


# Importações

In [ ]:
# Importações
import pandas as pd
import numpy as np
from pathlib import Path

# Carregando a base de dados

- A leitura parte do arquivo gravado no fim do notebook 01, com as colunas que ele selecionou
para a modelagem. As linhas estão na **mesma ordem da base bruta**: a linha *i* daqui é o
cliente da linha *i* de `data/raw`.


In [ ]:
# Carregando a base de dados.
df = pd.read_csv('../data/processed/Telcon-Customer-Churn_columns_selected.csv', sep=',')
linhas_iniciais = len(df)

# Registro de todos os ajustes feitos neste notebook.
ajustes = []

def registrar(ajuste, motivo, registros):
    ajustes.append({'ajuste': ajuste, 'motivo': motivo, 'registros_afetados': int(registros)})

df.shape

# `Churn` já retirada

O projeto é de **aprendizado não supervisionado**: trabalhamos como se a informação de
quem cancelou **não existisse**. O notebook 01 já retirou `Churn` na seleção de
características; aqui apenas confirmamos que ela não está na base, e nenhuma decisão deste
notebook (nem dos seguintes) é tomada olhando para ela.

> A alternativa (guardar `Churn` como rótulo reservado, só para avaliar os segmentos no
> final, como o professor faz com `preco_venda` no projeto de exemplo) está registrada em
> `docs/informacoes_complementares/decisao-churn.md` para discussão do grupo.


In [ ]:
# Churn foi retirada no notebook 01
assert 'Churn' not in df.columns
registrar('Churn retirada (notebook 01)',
          'Projeto não supervisionado: trabalhamos como se o rótulo não existisse',
          0)
df.shape


# 1. Registros duplicados

Clientes diferentes podem ter exatamente as mesmas características: são 15 colunas
categóricas e boa parte dos clientes novos tem `tenure` baixo e o mesmo pacote. A
pergunta é se as linhas repetidas são **o mesmo cliente contado duas vezes** (erro) ou
**clientes distintos com perfil igual** (dado legítimo).

In [ ]:
# Linhas idênticas em todas as colunas
duplicadas = df[df.duplicated(keep=False)]
print(f'{df.duplicated().sum()} linhas repetem outra linha da base')
duplicadas['tenure'].describe()

As 73 linhas envolvidas são **todas de clientes com `tenure = 1`**: primeiro mês, com o
mesmo pacote e a mesma mensalidade. É exatamente o perfil em que clientes diferentes
coincidem: quem acabou de entrar no plano básico ainda não tem histórico que o diferencie.
Na base bruta, cada uma dessas linhas tem um `customerID` diferente. Tratamos esses
registros como **clientes distintos com o mesmo perfil**: nenhum registro é removido.


In [ ]:
registrar('Duplicados mantidos',
          'Linhas iguais, todas com tenure 1: clientes novos diferentes com o mesmo perfil',
          0)

# 2. `TotalCharges` vazia

Na base bruta a coluna chegava como texto porque 11 clientes têm um espaço em branco no
lugar do valor; o notebook 01 já a converteu para número, e esses 11 viraram células vazias.
O ponto é entender **por que** o valor falta antes de decidir como preenchê-lo.


In [ ]:
# Os vazios chegam como NaN (conversão feita no notebook 01)
vazios = df['TotalCharges'].isna()
print(f'{vazios.sum()} clientes sem TotalCharges')
df.loc[vazios, ['tenure', 'MonthlyCharges', 'Contract']]

Os 11 clientes têm `tenure = 0`: entraram no mês da extração (10 deles já com contrato de
dois anos) e **ainda não receberam nenhuma cobrança**. Para o resto da base, `TotalCharges` acompanha `tenure × MonthlyCharges`
(a razão abaixo tem mediana 1), então o valor coerente para quem tem zero meses é **zero**.
Não é um valor desconhecido; é um valor que ainda não existe.

In [ ]:
# Evidência: TotalCharges ≈ tenure × MonthlyCharges nos clientes com valor
razao = df['TotalCharges'] / (df['tenure'] * df['MonthlyCharges'])
print(razao.describe().round(3))

# Todos os vazios são clientes sem nenhum mês de casa
assert (df.loc[vazios, 'tenure'] == 0).all()

df.loc[vazios, 'TotalCharges'] = 0.0
registrar('TotalCharges vazio = 0',
          'Clientes com tenure 0 ainda não foram cobrados (TotalCharges ≈ tenure × MonthlyCharges)',
          vazios.sum())

A razão varia entre ~0,7 e ~1,6 porque a mensalidade muda ao longo do contrato
(upgrades, descontos, reajustes). Isso é comportamento esperado e **não** é corrigido.

# 3. `SeniorCitizen` codificada como número

É a única variável binária guardada como `0/1`; todas as outras usam `Yes/No`. Por estar
como número, a análise de outliers do notebook 01 a apontou como discrepante (todo `1`
cai fora das cercas de Tukey, porque 84% dos clientes são `0`). Isso é **artefato de
codificação**, não cliente atípico. Ela passa para `Yes/No`, igual às demais.

In [ ]:
print(df['SeniorCitizen'].value_counts(normalize=True).round(3))

df['SeniorCitizen'] = df['SeniorCitizen'].map({0: 'No', 1: 'Yes'})
registrar('SeniorCitizen 0/1 → No/Yes',
          'Padronizar binárias; o 1 aparecia como outlier só por estar em escala numérica',
          len(df))

# 4. Categorias estruturais

`No internet service` e `No phone service` não são uma terceira resposta: repetem o que
`InternetService == "No"` e `PhoneService == "No"` já dizem. Mantê-las faz a mesma
informação entrar **sete vezes** na modelagem e infla as associações entre os serviços
(exatamente o que se vê no V de Cramér da EDA).

Primeiro, a prova de que a correspondência é exata:

In [ ]:
servicos_internet = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                     'TechSupport', 'StreamingTV', 'StreamingMovies']

# Cada "No internet service" deve corresponder exatamente a InternetService == "No"
sem_internet = df['InternetService'].eq('No')
for col in servicos_internet:
    assert (df[col].eq('No internet service') == sem_internet).all(), col

# E cada "No phone service" a PhoneService == "No"
sem_telefone = df['PhoneService'].eq('No')
assert (df['MultipleLines'].eq('No phone service') == sem_telefone).all()

print(f'{sem_internet.sum()} clientes sem internet e {sem_telefone.sum()} sem telefone')

In [ ]:
# Recodificando para "No": a ausência do serviço continua em InternetService/PhoneService
estruturais = {'No internet service': 'No', 'No phone service': 'No'}
colunas_estruturais = servicos_internet + ['MultipleLines']

tocados = df[colunas_estruturais].isin(estruturais.keys()).any(axis=1).sum()
df[colunas_estruturais] = df[colunas_estruturais].replace(estruturais)

registrar('No internet/phone service → No',
          'Categoria redundante com InternetService/PhoneService == "No"',
          tocados)

# 5. Valores discrepantes

As variáveis numéricas que sobraram (`tenure`, `MonthlyCharges`, `TotalCharges`) não têm
nenhum valor fora das cercas de Tukey, como a tabela `docs/tabelas/outliers.csv` já
mostrava: o único "outlier" era `SeniorCitizen`, resolvido no passo 3. Conferindo de novo:

In [ ]:
# Criando função para calcular os limites de outlier pelo método IQR
def limites_IQR(data):
    """Retorna os limites abaixo/acima dos quais um valor é considerado outlier."""
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
for col in numericas:
    inf, sup = limites_IQR(df[col])
    fora = ((df[col] < inf) | (df[col] > sup)).sum()
    print(f'{col:15s} limites [{inf:8.2f}, {sup:8.2f}]  fora: {fora}')

df[numericas].skew().round(2)

Nenhum registro precisa ser removido. `TotalCharges` tem assimetria à direita (~0,96),
o que é esperado (cresce com o tempo de casa), e deve ser tratada na **preparação**
(notebook 03: escala/transformação), não aqui. Este notebook corrige dados, não muda a
escala deles.

# 6. Tipos das colunas

Com os textos padronizados, as colunas categóricas passam para o tipo `category`.

In [ ]:
categoricas = df.select_dtypes(exclude='number').columns.tolist()
df[categoricas] = df[categoricas].astype('category')

# Conferência final: nada vazio, nenhum cliente perdido
assert df.notna().all().all()
assert len(df) == linhas_iniciais
df.info()

In [ ]:
# Categorias finais de cada coluna categórica
pd.Series({col: list(df[col].cat.categories) for col in categoricas})

# Registro dos ajustes

In [ ]:
tabela_ajustes = pd.DataFrame(ajustes)
tabela_ajustes.to_csv('../docs/tabelas/ajustes-dados.csv', index=False)
tabela_ajustes

# Salvando a base curada

Seguindo o padrão do modelo do professor (`..._tratado.csv`), a base vai para
`data/processed/Telcon-Customer-Churn_tratado.csv` com separador `;`. Sem `customerID`, a
ligação com o cliente é a **posição da linha**, a mesma da base bruta. O notebook 03 deve
ler com:

```python
df = pd.read_csv('../data/processed/Telcon-Customer-Churn_tratado.csv', sep=',')
```


In [ ]:
df.to_csv('../data/processed/Telcon-Customer-Churn_tratado.csv', sep=',', index=False)
print(f'{len(df)} clientes, {df.shape[1]} colunas gravadas')